In [1]:
import os
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
from shapely.strtree import STRtree
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")

# 1. Load H3 Static Features and Survival Predictions
df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))
df_surv = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "survival_predictions_grid.parquet"))
df_h3 = df_static.merge(df_surv[["h3_id", "predicted_burn_time_min", "burn_prob_3h", "burn_prob_6h", "burn_prob_12h"]], on="h3_id")

# Create GeoDataFrame for H3 polygons
gdf_h3_poly = gpd.GeoDataFrame(
    df_h3,
    geometry=gpd.GeoSeries.from_wkt(df_h3["geometry_wkt"]),
    crs="EPSG:4326"
)

# 2. Build Butte County Arterial & Evacuation Road Corridor Network
# Major nodes: Paradise, Magalia, Chico, Oroville, Concow, Forest Ranch
nodes_data = [
    {"node_id": 1001, "name": "Paradise_Center", "lon": -121.611, "lat": 39.759},
    {"node_id": 1002, "name": "Magalia_Upper", "lon": -121.578, "lat": 39.812},
    {"node_id": 1003, "name": "Chico_East", "lon": -121.780, "lat": 39.750},
    {"node_id": 1004, "name": "Chico_Downtown", "lon": -121.840, "lat": 39.730},
    {"node_id": 1005, "name": "Oroville_North", "lon": -121.560, "lat": 39.540},
    {"node_id": 1006, "name": "Oroville_Center", "lon": -121.556, "lat": 39.512},
    {"node_id": 1007, "name": "Forest_Ranch_Jct", "lon": -121.672, "lat": 39.881},
    {"node_id": 1008, "name": "Concow_Penta_Jct", "lon": -121.520, "lat": 39.770},
    {"node_id": 1009, "name": "Skyway_Midpoint", "lon": -121.690, "lat": 39.755},
    {"node_id": 1010, "name": "Clark_Rd_Jct", "lon": -121.590, "lat": 39.670},
    {"node_id": 1011, "name": "CA99_Skyway_Jct", "lon": -121.750, "lat": 39.710},
    {"node_id": 1012, "name": "CA70_North_Jct", "lon": -121.540, "lat": 39.620},
    {"node_id": 1013, "name": "Neal_Rd_Jct", "lon": -121.670, "lat": 39.680},
    {"node_id": 1014, "name": "Cohasset_Ridge_Jct", "lon": -121.731, "lat": 39.924}
]

df_nodes = pd.DataFrame(nodes_data)
node_coords = {row["node_id"]: (row["lon"], row["lat"]) for _, row in df_nodes.iterrows()}

# Corridor connections
corridors = [
    (1001, 1009, "Skyway_West", "primary", 2, 72.0),
    (1009, 1011, "Skyway_Lower", "primary", 2, 88.0),
    (1011, 1003, "Skyway_Chico_Connector", "primary", 3, 80.0),
    (1003, 1004, "Chico_Arterial", "primary", 3, 65.0),
    (1001, 1002, "Skyway_North", "secondary", 2, 60.0),
    (1002, 1007, "Skyway_High_Ridge", "secondary", 1, 55.0),
    (1007, 1014, "Cohasset_Forest_Rd", "secondary", 1, 50.0),
    (1014, 1003, "Cohasset_Chico_Route", "secondary", 2, 65.0),
    (1001, 1010, "Clark_Rd_South", "primary", 2, 80.0),
    (1010, 1012, "Clark_Oroville_Arterial", "primary", 2, 88.0),
    (1012, 1005, "CA70_South", "motorway", 2, 105.0),
    (1005, 1006, "Oroville_Access", "primary", 2, 70.0),
    (1001, 1013, "Neal_Rd_Escape", "tertiary", 1, 60.0),
    (1013, 1011, "Neal_CA99_Link", "tertiary", 2, 75.0),
    (1001, 1008, "Concow_Connector", "secondary", 1, 55.0),
    (1008, 1012, "Pentz_Rd_South", "secondary", 2, 70.0),
    (1011, 1005, "CA99_CA70_Crosslink", "primary", 2, 90.0)
]

# Generate bidirectional edges
edges_records = []
edge_id_counter = 1

def haversine_m(lon1, lat1, lon2, lat2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    return 6371000.0 * 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

for u, v, name, hwy, lanes, speed_kmh in corridors:
    # Forward direction
    u_lon, u_lat = node_coords[u]
    v_lon, v_lat = node_coords[v]
    length_m = haversine_m(u_lon, u_lat, v_lon, v_lat)
    geom = LineString([(u_lon, u_lat), (v_lon, v_lat)])
    
    # Capacity: lanes * 1800 for freeway/primary, 1000 for secondary/tertiary
    cap_per_lane = 1900 if hwy in ["motorway", "primary"] else 1100
    capacity_veh_hr = lanes * cap_per_lane
    base_time_s = length_m / (speed_kmh * (1000.0 / 3600.0))
    
    # Forward Edge
    edges_records.append({
        "edge_id": edge_id_counter,
        "u": u,
        "v": v,
        "name": f"{name}_fwd",
        "highway_type": hwy,
        "lanes": np.int16(lanes),
        "speed_kmh": np.float32(speed_kmh),
        "length_m": np.float32(length_m),
        "capacity_veh_hr": np.int32(capacity_veh_hr),
        "base_time_s": np.float32(base_time_s),
        "geometry": geom
    })
    edge_id_counter += 1
    
    # Reverse Edge
    edges_records.append({
        "edge_id": edge_id_counter,
        "u": v,
        "v": u,
        "name": f"{name}_rev",
        "highway_type": hwy,
        "lanes": np.int16(lanes),
        "speed_kmh": np.float32(speed_kmh),
        "length_m": np.float32(length_m),
        "capacity_veh_hr": np.int32(capacity_veh_hr),
        "base_time_s": np.float32(base_time_s),
        "geometry": LineString([(v_lon, v_lat), (u_lon, u_lat)])
    })
    edge_id_counter += 1

gdf_edges = gpd.GeoDataFrame(edges_records, geometry="geometry", crs="EPSG:4326")
print(f"Constructed network: {len(df_nodes)} nodes, {len(gdf_edges)} directed edges.")

# 3. Spatial STRtree Join: Map Road Edges to Intersecting H3 Risk Cells
h3_geoms = gdf_h3_poly.geometry.values
strtree = STRtree(h3_geoms)

edge_primary_h3 = []
edge_burn_p3 = []
edge_burn_p6 = []
edge_burn_p12 = []
edge_pm25 = []

for _, edge in gdf_edges.iterrows():
    # Query STRtree using edge LineString
    intersecting_indices = strtree.query(edge.geometry)
    
    if len(intersecting_indices) > 0:
        matched_cells = gdf_h3_poly.iloc[intersecting_indices]
        # Primary risk cell: cell with highest 12h burn risk along the road segment
        worst_cell = matched_cells.sort_values("burn_prob_12h", ascending=False).iloc[0]
        
        edge_primary_h3.append(np.uint64(worst_cell["h3_id"]))
        edge_burn_p3.append(np.float32(worst_cell["burn_prob_3h"]))
        edge_burn_p6.append(np.float32(worst_cell["burn_prob_6h"]))
        edge_burn_p12.append(np.float32(worst_cell["burn_prob_12h"]))
        edge_pm25.append(np.float32(worst_cell["pm25_baseline_ugm3"]))
    else:
        # Fallback to centroid nearest neighbor
        edge_midpoint = edge.geometry.interpolate(0.5, normalized=True)
        nearest_idx = strtree.nearest(edge_midpoint)
        nearest_cell = gdf_h3_poly.iloc[nearest_idx]
        
        edge_primary_h3.append(np.uint64(nearest_cell["h3_id"]))
        edge_burn_p3.append(np.float32(nearest_cell["burn_prob_3h"]))
        edge_burn_p6.append(np.float32(nearest_cell["burn_prob_6h"]))
        edge_burn_p12.append(np.float32(nearest_cell["burn_prob_12h"]))
        edge_pm25.append(np.float32(nearest_cell["pm25_baseline_ugm3"]))

gdf_edges["primary_h3_id"] = edge_primary_h3
gdf_edges["burn_prob_t3"] = edge_burn_p3
gdf_edges["burn_prob_t6"] = edge_burn_p6
gdf_edges["burn_prob_t12"] = edge_burn_p12
gdf_edges["pm25_ugm3"] = edge_pm25

print("\n--- Spatial Graph-Risk Fusion Summary ---")
print(f"Total directed edges: {len(gdf_edges)}")
print(f"Mean road length: {gdf_edges['length_m'].mean():.1f} meters")
print(f"Max 12h Burn Probability across road network: {gdf_edges['burn_prob_t12'].max():.4f}")
print(f"Mean baseline road travel time: {gdf_edges['base_time_s'].mean():.1f} seconds")

Constructed network: 14 nodes, 34 directed edges.

--- Spatial Graph-Risk Fusion Summary ---
Total directed edges: 34
Mean road length: 9742.8 meters
Max 12h Burn Probability across road network: 0.9998
Mean baseline road travel time: 496.9 seconds


In [2]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pickle

# 1. BPR Calibration Constants
ALPHA = 0.15
GAMMA = 4.0
BETA = 3600.0          # Up to 1 hr penalty as fire probability approaches critical
DELTA = 2.0            # 2s per ug/m3 PM2.5 visibility penalty
TAU_CRITICAL = 0.75    # 75% burn probability triggers physical road closure
INF_WEIGHT = 1e9

# 2. Compute Time-Stepped Dynamic Weights
def compute_edge_weight(base_time, capacity, volume, burn_prob, pm25):
    if burn_prob >= TAU_CRITICAL:
        return np.float32(INF_WEIGHT)
    
    # BPR congestion term
    v_c_ratio = volume / max(1.0, float(capacity))
    congestion_factor = 1.0 + ALPHA * (v_c_ratio ** GAMMA)
    
    # Fire and smoke penalties
    fire_penalty = BETA * burn_prob
    smoke_penalty = DELTA * max(0.0, pm25 - 35.0)  # Penalty kicks in above moderate AQI
    
    total_cost_s = (base_time * congestion_factor) + fire_penalty + smoke_penalty
    return np.float32(total_cost_s)

# Horizon weights (assuming initial baseline traffic volume = 0 for free-flow evacuation start)
weight_t0 = []
weight_t3 = []
weight_t6 = []
weight_t12 = []

for _, row in gdf_edges.iterrows():
    bt = row["base_time_s"]
    cap = row["capacity_veh_hr"]
    pm = row["pm25_ugm3"]
    
    w0 = compute_edge_weight(bt, cap, 0.0, 0.0, pm)
    w3 = compute_edge_weight(bt, cap, 0.0, row["burn_prob_t3"], pm)
    w6 = compute_edge_weight(bt, cap, 0.0, row["burn_prob_t6"], pm)
    w12 = compute_edge_weight(bt, cap, 0.0, row["burn_prob_t12"], pm)
    
    weight_t0.append(w0)
    weight_t3.append(w3)
    weight_t6.append(w6)
    weight_t12.append(w12)

gdf_edges["weight_t0_s"] = weight_t0
gdf_edges["weight_t3_s"] = weight_t3
gdf_edges["weight_t6_s"] = weight_t6
gdf_edges["weight_t12_s"] = weight_t12

# Check closures
closures_t3 = (gdf_edges["weight_t3_s"] >= INF_WEIGHT).sum()
closures_t6 = (gdf_edges["weight_t6_s"] >= INF_WEIGHT).sum()
closures_t12 = (gdf_edges["weight_t12_s"] >= INF_WEIGHT).sum()

print("\n--- Dynamic Edge Cost & Closures Summary ---")
print(f"Total Road Edges: {len(gdf_edges)}")
print(f"Road Closures @ T=0h:  0 / {len(gdf_edges)}")
print(f"Road Closures @ T=3h:  {closures_t3} / {len(gdf_edges)}")
print(f"Road Closures @ T=6h:  {closures_t6} / {len(gdf_edges)}")
print(f"Road Closures @ T=12h: {closures_t12} / {len(gdf_edges)}")

# 3. Export Parquet Tables
df_edges_export = pd.DataFrame(gdf_edges.drop(columns=["geometry"]))
df_edges_export["geometry_wkt"] = gdf_edges.geometry.apply(lambda g: g.wkt)

nodes_parquet_path = os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet")
edges_parquet_path = os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet")

df_nodes.to_parquet(nodes_parquet_path, compression="snappy")
df_edges_export.to_parquet(edges_parquet_path, compression="snappy")

# 4. Build and Pickle Graph Object (Rustworkx or NetworkX fallback)
has_rustworkx = False
try:
    import rustworkx as rx
    has_rustworkx = True
except ImportError:
    pass

if has_rustworkx:
    print("\nAssembling PyDiGraph via Rustworkx...")
    graph = rx.PyDiGraph()
    node_indices = {}
    for _, n in df_nodes.iterrows():
        idx = graph.add_node({"node_id": n["node_id"], "name": n["name"], "lon": n["lon"], "lat": n["lat"]})
        node_indices[n["node_id"]] = idx
        
    for _, e in df_edges_export.iterrows():
        u_idx = node_indices[e["u"]]
        v_idx = node_indices[e["v"]]
        edge_data = {
            "edge_id": e["edge_id"],
            "length_m": e["length_m"],
            "base_time_s": e["base_time_s"],
            "capacity": e["capacity_veh_hr"],
            "weights": [e["weight_t0_s"], e["weight_t3_s"], e["weight_t6_s"], e["weight_t12_s"]]
        }
        graph.add_edge(u_idx, v_idx, edge_data)
else:
    import networkx as nx
    print("\nAssembling DiGraph via NetworkX fallback...")
    graph = nx.DiGraph()
    for _, n in df_nodes.iterrows():
        graph.add_node(n["node_id"], name=n["name"], lon=n["lon"], lat=n["lat"])
    for _, e in df_edges_export.iterrows():
        graph.add_edge(
            e["u"], e["v"],
            edge_id=e["edge_id"],
            length_m=e["length_m"],
            base_time_s=e["base_time_s"],
            capacity=e["capacity_veh_hr"],
            weights=[e["weight_t0_s"], e["weight_t3_s"], e["weight_t6_s"], e["weight_t12_s"]]
        )

graph_pkl_path = os.path.join(PROCESSED_DATA_DIR, "road_graph.pkl")
with open(graph_pkl_path, "wb") as f:
    pickle.dump(graph, f)

print(f"\nSaved graph artifacts:")
print(f" - {nodes_parquet_path}")
print(f" - {edges_parquet_path}")
print(f" - {graph_pkl_path}")


--- Dynamic Edge Cost & Closures Summary ---
Total Road Edges: 34
Road Closures @ T=0h:  0 / 34
Road Closures @ T=3h:  20 / 34
Road Closures @ T=6h:  24 / 34
Road Closures @ T=12h: 32 / 34

Assembling DiGraph via NetworkX fallback...

Saved graph artifacts:
 - C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\road_nodes.parquet
 - C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\road_edges.parquet
 - C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\road_graph.pkl
